# Resultados de ciencia de datos internacional
Consolida el experimento reproducible de caracteristicas, seleccion temporal, pruebas estadisticas y validacion por fuente.

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL_DIR = PROJECT_ROOT / 'data' / '04_model_output' / 'modeling'
VALIDATION_DIR = PROJECT_ROOT / 'data' / '04_model_output' / 'validation'
FEATURES = PROJECT_ROOT / 'data' / '04_model_output' / 'features' / 'tsi_daily_features.parquet'
metrics = pd.read_csv(MODEL_DIR / 'metricas_modelos.csv')
tests = pd.read_csv(MODEL_DIR / 'pruebas_modelos_vs_baseline.csv')
by_source = pd.read_csv(VALIDATION_DIR / 'metricas_por_fuente.csv')
bootstrap = pd.read_csv(VALIDATION_DIR / 'intervalo_bootstrap_mejora.csv')
features = pd.read_parquet(FEATURES)
display(metrics)
display(tests)
display(by_source)
display(bootstrap)

,modelo,mae,rmse,r2
0,random_forest,2.511087,5.274513,0.947871
1,ridge,3.514525,6.338738,0.924712
2,baseline_persistencia,5.302138,9.950087,0.814488


,modelo,mejora_mae_vs_baseline,t_pareada_p_valor,wilcoxon_p_valor
0,ridge,1.787612,9.673231e-31,1.372203e-21
1,random_forest,2.791051,2.731364e-64,1.632495e-113


,fuente_original,filas_prueba,mae_baseline,mae_random_forest,rmse_baseline,rmse_random_forest,mejora_mae,mejora_relativa_pct
0,istanbul_traffic_index,665.0,12.113811,5.370551,16.285460,8.305310,6.743260,55.665883
1,madrid_traffic_points,5.0,0.248895,0.946630,0.306946,0.991827,-0.697736,-280.333651
2,metr_la,23.0,12.689391,10.807494,21.619667,16.364016,1.881897,14.830472
3,pems_bay,35.0,3.112371,0.624970,3.749540,1.081918,2.487401,79.919814
4,sao_paulo_cet,1218.0,1.527293,0.853842,2.036309,1.301546,0.673451,44.094419


,mejora_media_mae,ic95_inferior,ic95_superior,proporcion_muestras_mejoradas,fuentes_con_mejora,fuentes_evaluadas
0,2.791051,2.481661,3.089033,0.734841,4,5


In [3]:
selected = metrics.sort_values('mae').iloc[0]
summary = {
    'tarea': 'pronostico_del_tsi_provisional_del_siguiente_dia',
    'filas_features': int(len(features)),
    'modelo_candidato': str(selected['modelo']),
    'mae_prueba_modelo_candidato': float(selected['mae']),
    'mae_prueba_baseline': float(metrics.loc[metrics['modelo'] == 'baseline_persistencia', 'mae'].iloc[0]),
    'fuentes_con_mejora': int((by_source['mejora_mae'] > 0).sum()),
    'fuentes_evaluadas': int(len(by_source)),
    'excepciones': by_source.loc[by_source['mejora_mae'] <= 0, 'fuente_original'].tolist(),
    'estado': 'candidato_internacional_no_final',
    'limitaciones': ['TSI provisional', 'Madrid tiene cinco filas de prueba', 'sin etiqueta de pre_colapso', 'sin calibracion local de Guadalajara'],
}
output = PROJECT_ROOT / 'data' / '04_model_output' / 'results'
output.mkdir(parents=True, exist_ok=True)
(output / 'resumen_experimento_internacional.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "tarea": "pronostico_del_tsi_provisional_del_siguiente_dia",
  "filas_features": 9742,
  "modelo_candidato": "random_forest",
  "mae_prueba_modelo_candidato": 2.5110870774445324,
  "mae_prueba_baseline": 5.30213794644081,
  "fuentes_con_mejora": 4,
  "fuentes_evaluadas": 5,
  "excepciones": [
    "madrid_traffic_points"
  ],
  "estado": "candidato_internacional_no_final",
  "limitaciones": [
    "TSI provisional",
    "Madrid tiene cinco filas de prueba",
    "sin etiqueta de pre_colapso",
    "sin calibracion local de Guadalajara"
  ]
}


In [4]:
display(pd.DataFrame([
    {'decision': 'modelo_candidato', 'valor': summary['modelo_candidato']},
    {'decision': 'mejora_media_mae', 'valor': float(bootstrap['mejora_media_mae'].iloc[0])},
    {'decision': 'intervalo_bootstrap_95', 'valor': f"[{bootstrap['ic95_inferior'].iloc[0]:.3f}, {bootstrap['ic95_superior'].iloc[0]:.3f}]"},
    {'decision': 'estabilidad_por_fuente', 'valor': f"{summary['fuentes_con_mejora']}/{summary['fuentes_evaluadas']}"},
    {'decision': 'estado', 'valor': summary['estado']},
]))

,decision,valor
0,modelo_candidato,random_forest
1,mejora_media_mae,2.791051
2,intervalo_bootstrap_95,"[2.482, 3.089]"
3,estabilidad_por_fuente,4/5
4,estado,candidato_internacional_no_final


## Decision
El Random Forest queda como candidato internacional para pronosticar el TSI provisional del siguiente dia. La decision no autoriza una etiqueta de pre-colapso ni una publicacion como TSI final; requiere ampliar la evaluacion de Madrid, documentar una etiqueta operacional y realizar la validacion local futura.